#### 1.

In [1]:
import sys
import os

# Add the project root to the path so we can import 'src'
# Assuming our notebook is in 'recommender-ml10m/notebooks/'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import DataLoader

# Initialize with a path relative to the PROJECT ROOT
# This 'data' folder will be created in our main project directory
loader = DataLoader(base_path=os.path.join(project_root, 'data'))

# Download, extract, and load the dataset
df = loader.get_processed_data()

print(f"Data shape: {df.shape}")
print(df.head())

Merging data...
Converting timestamps...
Data shape: (10000054, 6)
   userId  movieId  rating                 title  \
0       1      122     5.0      Boomerang (1992)   
1       1      185     5.0       Net, The (1995)   
2       1      231     5.0  Dumb & Dumber (1994)   
3       1      292     5.0       Outbreak (1995)   
4       1      316     5.0       Stargate (1994)   

                         genres  year  
0                Comedy|Romance  1996  
1         Action|Crime|Thriller  1996  
2                        Comedy  1996  
3  Action|Drama|Sci-Fi|Thriller  1996  
4       Action|Adventure|Sci-Fi  1996  


#### 2.

In [2]:
from src.analyzer import GenreAnalyzer

analyzer = GenreAnalyzer(df)
summary = analyzer.analyze_trends()

# --- View 1: The Raw Exploded Data (Massive Table) ---
# Be careful with .head(), this table has ~25 million rows!
print("Exploded View Sample:")
display(analyzer.exploded_df.head(15))

# --- View 2: The Yearly Averages for EVERY genre/year combination ---
print("\nAnnual Stats Table:")
display(analyzer.annual_stats.sort_values(['genre', 'year']))

# --- View 3: The Full Results (Not just top 5) ---
print("\nFull Genre Trend Summary:")
display(analyzer.summary_df)

# --- Insight for Step 3 ---
# Look for genres with a small 'start_count' in 'summary_df'.
# These are the ones where the 'earliest year' likely skewed the results.

Exploding genres...
Exploded View Sample:


,year,rating,genres,genre
0,1996,5.0,Comedy|Romance,Comedy
0,1996,5.0,Comedy|Romance,Romance
1,1996,5.0,Action|Crime|Thriller,Action
1,1996,5.0,Action|Crime|Thriller,Crime
1,1996,5.0,Action|Crime|Thriller,Thriller
2,1996,5.0,Comedy,Comedy
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Action
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Drama
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Sci-Fi
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Thriller



Annual Stats Table:


,genre,year,avg_rating,rating_count
0,(no genres listed),2004,2.750000,2
1,(no genres listed),2007,4.000000,3
2,(no genres listed),2008,4.000000,2
3,Action,1995,3.000000,1
4,Action,1996,3.440509,331387
...,...,...,...,...
271,Western,2005,3.460447,23753
272,Western,2006,3.525736,14940
273,Western,2007,3.529675,11845
274,Western,2008,3.598341,12660



Full Genre Trend Summary:


,genre,start_year,end_year,start_rating,end_rating,start_count,decrease
11,Horror,1995,2009,5.000000,3.310457,1,1.689543
17,Thriller,1995,2009,5.000000,3.472470,1,1.527530
14,Mystery,1995,2009,5.000000,3.659091,1,1.340909
6,Crime,1995,2009,4.000000,3.619338,2,0.380662
18,War,1996,2009,3.943008,3.723529,59973,0.219478
7,Documentary,1996,2009,3.867188,3.682051,4736,0.185136
15,Romance,1996,2009,3.617932,3.443820,231765,0.174112
13,Musical,1996,2009,3.631460,3.466177,55378,0.165284
4,Children,1996,2009,3.531240,3.383498,102193,0.147742
3,Animation,1996,2009,3.688559,3.572374,56977,0.116184


#### 3. The decline in genre popularity is absolutely affected by the number of ratings.
 * The Problem: In the early years of the dataset (late 90s), some niche genres might have only 10 ratings. If those 10 people were superfans, the average might be 4.5. Ten years later, with 100,000 ratings from the general public, the average might drop to 3.5.

The Conclusion: The "decrease" might not be a drop in quality or popularity, but simply a regression to the mean as the sample size increased. By adding a min_ratings threshold, we ensure the "first year" and "last year" are both statistically significant.

#### 4.